# 03 — Delivering and Evaluating a Grounded Solution

**Module 3.** TO: *Deliver a RAG-based solution that produces grounded,
relevant responses.*

Uses the same `state-driver-manuals` index Module 2 built — run `02` first.

**Two chat deployments, for two different reasons:**
`gpt-4.1-mini` powers Clip 1's managed "on your data" call, because that
Azure OpenAI extension still sends the legacy `max_tokens` parameter
internally and newer reasoning-class models (`gpt-5-mini`) reject it with
`Unsupported parameter: 'max_tokens'`. `gpt-5-mini` powers Clip 2's
hand-rolled pipeline and Clip 3's evaluators, since those are plain chat
completions you construct yourself with no such constraint. This is exactly
the kind of SDK/API surface drift to watch for.


## Setup — run once per kernel session

In [ ]:
from azure.identity import DefaultAzureCredential, get_bearer_token_provider
from azure.search.documents import SearchClient
from azure.search.documents.models import VectorizableTextQuery
from openai import AzureOpenAI

import sys
sys.path.insert(0, "..")

from config import load_config

config = load_config()
credential = DefaultAzureCredential()
search_client = SearchClient(config.search_endpoint, config.search_index_name, credential)

token_provider = get_bearer_token_provider(credential, "https://cognitiveservices.azure.com/.default")
openai_client = AzureOpenAI(
    azure_endpoint=config.openai_endpoint,
    azure_ad_token_provider=token_provider,
    api_version="2024-10-21",
)

ONYOURDATA_DEPLOYMENT = config.openai_onyourdata_deployment  # gpt-4.1-mini
CHAT_DEPLOYMENT = config.openai_chat_deployment  # gpt-5-mini

HERO_QUESTION = "What's the speed limit in a school zone?"

print("Search index:", config.search_index_name)
print("On-your-data deployment:", ONYOURDATA_DEPLOYMENT)
print("Hand-rolled/judge deployment:", CHAT_DEPLOYMENT)


---
## Clip 1 (2 min) — Connecting the Index to a Model in Foundry

The managed path: attach the Search index as a data source on the chat
call itself (Azure OpenAI "on your data"). Authentication is
`system_assigned_managed_identity` — the AI Services resource's own
identity calls Search, granted `Search Index Data Reader` and
`Search Service Contributor` on the Search service. No API keys anywhere
in this cell.


In [ ]:
def ask_with_managed_data_source(query, top_n=5):
    response = openai_client.chat.completions.create(
        model=ONYOURDATA_DEPLOYMENT,
        messages=[{"role": "user", "content": query}],
        extra_body={
            "data_sources": [{
                "type": "azure_search",
                "parameters": {
                    "endpoint": config.search_endpoint,
                    "index_name": config.search_index_name,
                    "authentication": {"type": "system_assigned_managed_identity"},
                    "query_type": "vector_semantic_hybrid",
                    "semantic_configuration": "semantic-config",
                    "embedding_dependency": {
                        "type": "deployment_name",
                        "deployment_name": config.openai_embedding_deployment,
                    },
                    "top_n_documents": top_n,
                },
            }],
        },
    )
    message = response.choices[0].message
    citations = (message.model_extra or {}).get("context", {}).get("citations", [])
    return message.content, citations


In [ ]:
answer, citations = ask_with_managed_data_source(HERO_QUESTION)
print("Answer:")
print(answer)
print()
print(f"{len(citations)} citation(s) — click through and read the sentence yourself:")
for c in citations[:5]:
    print(f"  - {c.get('title')}: {c.get('content', '')[:150].strip()}...")


The managed path is the fastest way to ship — Foundry
decided chunk selection, the grounding instruction, and the citation
format for you. *"That's fine until you need to change one of those."*
Clip 2 answers that question rather than repeating this demo.


---
## Clip 2 (4 min) — Constructing the Grounded Prompt Pipeline

Rebuilt by hand so nothing is a black box: hybrid + semantic search selects
passages, a system prompt instructs the model to answer only from them and
say so when it can't, then a plain chat completion generates the answer.


In [ ]:
import random
import threading
import time

from openai import RateLimitError

# Clip 3 fans this out across all 20 eval rows concurrently via evaluate(),
# which bursts past the deployment's rate limit regardless of how much
# capacity you provision. A process-wide semaphore caps real concurrency
# (not just retries-after-the-fact) — a production pipeline behind any
# rate-limited API needs the same kind of throttle.
CHAT_SEMAPHORE = threading.Semaphore(3)

SYSTEM_PROMPT = (
    "Answer only using the passages in the context below. Cite the state "
    "each fact comes from. If the context does not contain the answer, say "
    "you don't have that information — do not use outside knowledge."
)


def retrieve_passages(query, state_filter=None, top=5):
    results = search_client.search(
        search_text=query,
        vector_queries=[VectorizableTextQuery(text=query, k_nearest_neighbors=top, fields="vector")],
        query_type="semantic",
        semantic_configuration_name="semantic-config",
        filter=f"state eq '{state_filter}'" if state_filter else None,
        top=top,
        select=["state", "title", "chunk"],
    )
    return [{"state": r["state"], "title": r["title"], "chunk": r["chunk"]} for r in results]


def format_context(passages):
    return "\n\n".join(f"[{p['state']}] {p['chunk']}" for p in passages)


def chat_completion_with_retry(**kwargs):
    with CHAT_SEMAPHORE:
        for attempt in range(8):
            try:
                return openai_client.chat.completions.create(**kwargs)
            except RateLimitError:
                if attempt == 7:
                    raise
                time.sleep(min(2 ** attempt, 30) + random.random())


def answer_question(query, state_filter=None, top=5):
    passages = retrieve_passages(query, state_filter=state_filter, top=top)
    context = format_context(passages)
    response = chat_completion_with_retry(
        model=CHAT_DEPLOYMENT,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {query}"},
        ],
    )
    return {"answer": response.choices[0].message.content, "passages": passages, "context": context}


In [ ]:
result = answer_question(HERO_QUESTION)
print("Answer:", result["answer"])
print()
print("Retrieved from:", [p["state"] for p in result["passages"]])


**The refusal.** Texas isn't indexed. The model very likely knows this
answer from pretraining — that's exactly why this demo works: a pipeline
that answers anyway is guessing, not grounded, and you'd have no way to
tell from the prose alone.


In [ ]:
refusal = answer_question("What's the school zone speed limit in Texas?")
print(refusal["answer"])


**The citation failure — the most valuable 60 seconds in the course.** Ask
a New Jersey question with the filter off and `top=1`. Watch for a fluent,
confident answer that cites a real document — just not New Jersey's.


In [ ]:
citation_failure = answer_question("How close to a fire hydrant can I park?", state_filter=None, top=1)
print(citation_failure["answer"])
print()
print("Actually retrieved from:", [p["state"] for p in citation_failure["passages"]])
print("Takeaway: nothing about that prose signals a problem — judge the source, not the sentence.")


---
## Clip 3 (4 min) — Measuring Groundedness and Relevance

Runs the ~20-row eval set through the same `answer_question()` pipeline,
**unfiltered** — the point is to see which rows an unfiltered pipeline gets
wrong, not to grade a pipeline you've already fixed. Scored with three
evaluators, in the order that matters: retrieval (did we find the right
state's passage?), groundedness (is the answer supported by what we
found?), relevance (does it answer the question?).


In [ ]:
from azure.ai.evaluation import evaluate, RetrievalEvaluator, GroundednessEvaluator, RelevanceEvaluator

model_config = {
    "azure_endpoint": config.openai_endpoint,
    "azure_deployment": CHAT_DEPLOYMENT,
    # No `credential` key: the SDK's own model_config validator has a bug
    # that rejects it outright (fails on the `credential: NotRequired[Any]`
    # annotation). Leaving both `credential` and `api_key` out isn't a
    # workaround so much as the actual answer — the judge calls still
    # authenticate keyless via DefaultAzureCredential under the hood.
}

# is_reasoning_model=True: gpt-5-mini rejects the legacy `max_tokens` param
# these evaluators send by default; this flag switches them to
# max_completion_tokens. Same drift as Clip 1's on-your-data extension.
retrieval_evaluator = RetrievalEvaluator(model_config=model_config, is_reasoning_model=True)
groundedness_evaluator = GroundednessEvaluator(model_config=model_config, is_reasoning_model=True)
relevance_evaluator = RelevanceEvaluator(model_config=model_config, is_reasoning_model=True)


def eval_target(query, state, ground_truth):
    # evaluate() inspects the target's parameter *names* against the
    # data's columns rather than accepting a **kwargs catch-all — a plain
    # **kwargs param is itself treated as a required column named "kwargs"
    # and the call fails. Name every column from eval_set.jsonl explicitly.
    #
    # state/ground_truth are unused here on purpose: retrieval is
    # deliberately run unfiltered, so a pipeline that ignores the state
    # filter scores badly on the rows where that matters, visibly.
    result = answer_question(query, state_filter=None, top=5)
    return {"response": result["answer"], "context": result["context"]}


In [ ]:
eval_result = evaluate(
    data="../eval/eval_set.jsonl",
    target=eval_target,
    evaluators={
        "retrieval": retrieval_evaluator,
        "groundedness": groundedness_evaluator,
        "relevance": relevance_evaluator,
    },
    evaluator_config={
        "retrieval": {"column_mapping": {"query": "${data.query}", "context": "${target.context}"}},
        "groundedness": {"column_mapping": {"query": "${data.query}", "response": "${target.response}", "context": "${target.context}"}},
        "relevance": {"column_mapping": {"query": "${data.query}", "response": "${target.response}"}},
    },
    output_path="../eval/eval_results.json",
    # If the Foundry project data-plane hasn't finished propagating yet
    # (can take longer than ARM's own "Succeeded" suggests — a real quirk
    # hit while building this notebook), this only skips the portal push;
    # the local scoring above still runs and still prints below.
    azure_ai_project=config.ai_project_endpoint or None,
)
print("Rows evaluated:", len(eval_result["rows"]))


In [ ]:
metrics = eval_result["metrics"]
for name, value in sorted(metrics.items()):
    print(f"{name}: {value}")


**The diagnostic beat.** Find a row with low retrieval but high
groundedness — the model faithfully quoted the wrong state's passage. Add
the state filter for that row, re-run just it, and watch the retrieval
score move while groundedness stays high (it was never the problem).


In [ ]:
rows = eval_result["rows"]
candidates = sorted(
    rows,
    key=lambda r: (r.get("outputs.retrieval.retrieval", 5), -r.get("outputs.groundedness.groundedness", 0)),
)
worst = candidates[0]
print("Lowest-retrieval row:")
print(" query:", worst.get("inputs.query"))
print(" expected state:", worst.get("inputs.state"))
print(" retrieval score:", worst.get("outputs.retrieval.retrieval"))
print(" groundedness score:", worst.get("outputs.groundedness.groundedness"))
print(" answer:", worst.get("outputs.response", "")[:200])


In [ ]:
# Re-run that one row with the state filter applied, then re-score it alone.
fixed = answer_question(worst["inputs.query"], state_filter=worst["inputs.state"], top=5)

retrieval_before = worst.get("outputs.retrieval.retrieval")
retrieval_after = retrieval_evaluator(query=worst["inputs.query"], context=fixed["context"])["retrieval"]
groundedness_after = groundedness_evaluator(query=worst["inputs.query"], response=fixed["answer"], context=fixed["context"])["groundedness"]

print(f"Retrieval score:    {retrieval_before} -> {retrieval_after}")
print(f"Groundedness score: {worst.get('outputs.groundedness.groundedness')} -> {groundedness_after}")
print()
print("Fixed answer:", fixed["answer"][:200])
print("Takeaway: groundedness alone would have certified the wrong answer as well-supported — retrieval is what tells you where the failure actually is.")


---
## Done

All three enabling objectives for Terminal Objective 3 covered: a managed
Foundry connection (Clip 1), a hand-built grounded pipeline that proves the
grounding constraint with a refusal and exposes a citation failure
(Clip 2), and a diagnostic evaluation that separates a retrieval problem
from a generation problem (Clip 3).
